In [103]:
# https://musicbrainz.org/doc/MusicBrainz_API
#
# See also:
# - https://musicbrainz.org/ws/2/artist?query=artist:%22Nathan%20Ross%22

In [104]:
# Uncomment if needed
# %pip install requests pandas python-dotenv rich

In [105]:
# Define Functions: Structure Normalization

def get_recordings_for_artist_relations_data(l_relations_data: list) -> list[dict]:
    """
    :param l_relations_data: e.g., returned by mb_get(f"artist/{artist_id}", ...)["relations"]
    :return: app-normalized lists of recordings
    """

    l_recordings = []

    for l_relation in l_relations_data:

        l_mb_type = l_relation["type"]
        if l_mb_type != "instrument":
            assert l_mb_type == "discogs"
            continue

        l_recording = l_relation["recording"]

        l_recordings.append({
            "begin": l_relation["begin"],
            "end": l_relation["end"],
            "title": l_recording["title"],
            "id": l_recording["id"],
            "credit": l_relation["source-credit"],
            "roles": l_relation["attributes"],
        })

    return l_recordings


def urls_for_relations_data(l_relations_data: list) -> set[str]:
    l_urls = set()
    for l_relation in l_relations_data:
        if l_relation.get("target-type") == "url":
            l_urls.add(l_relation["url"]["resource"])

    return l_urls


In [106]:
# Define Functions: Load and Parse Data using MusicBrainz APIs

from pprint import pprint
import requests

# retrieve info from MusixBrainz; see also:
# - https://musicbrainz.org/relationships
# - https://musicbrainz.org/doc/MusicBrainz_Entity
def mb_get(l_endpoint: str, **l_params):
    l_params["fmt"] = "json"

    r = requests.get(
        f"https://musicbrainz.org/ws/2/{l_endpoint}",
        params=l_params,
        headers={
            "User-Agent": "NathanRossQueryAgent/0.1 (noodnik2@gmail.com)"
        },
        timeout=30,
    )

    r.raise_for_status()
    return r.json()


def player_urls_for_recording(l_recording_mbid: str) -> set[str]:
    """
    Returns every player URL known to MusicBrainz for this recording.
    """

    l_recording_data = mb_get(
        f"recording/{l_recording_mbid}",
        inc="url-rels",
        # limit=1, # TODO: remove this temporary change to avoid throttling
    )

    return urls_for_relations_data(l_recording_data.get("relations", []))


In [107]:
# Find Nathan Ross in online MusicBrainz Database

loaded_artist_info = mb_get(
    "artist",
    query='artist:"Nathan Ross"',
    limit=50,
)

In [108]:
# Extract the single `artist_id` expected in MusicBrainz response

artists = loaded_artist_info["artists"]
assert len(artists) == 1
artist = artists[0]
artist_id = artist["id"]

In [109]:
# Load MusicBrainz recordings for the `artist` identified by `artist_id`

artist_recordings = mb_get(
    f"artist/{artist_id}",
    inc="recording-rels"
)

In [110]:
# Retrieve normalized `releases` and `recordings` for returned MusicBrainz `artist`

recordings = get_recordings_for_artist_relations_data(
    artist_recordings["relations"]
)

In [111]:
# Create Recordings Dataframe

import pandas as pd

recordings_pd = pd.DataFrame(recordings)
recordings_pd.head()

,begin,end,title,id,credit,roles
0,1953-01-20,1953-01-20,Blue Gardenia,fb0fd734-a880-4984-9dfb-011f7a7188fb,,[violin]
1,1954-07-27,1954-07-27,Smile,298da2a1-c0dc-46ad-a593-2977f5a8a883,,[violin]
2,1954-12-20,1954-12-20,A Blossom Fell,601a8791-3e90-49ea-884a-0b49bd5a38fd,,[violin]
3,1954-12-20,1954-12-20,If I May,1c8b86fd-57cc-4e21-955c-30fc93f06434,,[violin]
4,1954-12-20,1954-12-20,The Sand and the Sea,b3a96399-5670-43da-898d-e16b562cf1cf,,[violin]


In [113]:
# Load Recordings URLs
# recording_mbid=recordings[0]["id"]
recording_mbid="601a8791-3e90-49ea-884a-0b49bd5a38fd"
urls = player_urls_for_recording(recording_mbid)

print(
    [{
        "recording_mbid": recording_mbid,
        "urls": urls,
    }]
)

{'recording_mbid': '601a8791-3e90-49ea-884a-0b49bd5a38fd', 'urls': {'https://open.spotify.com/track/397rt4rZoEdxDaw3kCfOP6', 'https://www.deezer.com/track/3138421', 'https://open.spotify.com/track/2d78J2bnQWINeQj4tk3Ix2', 'https://secondhandsongs.com/performance/420396'}}
